# Validate Gold — CineData Analytics
Quality gate da camada Gold: roda regras que sempre devem valer (grão da fato, integridade das bridges, domínio de gêneros, notas, financeiro e contexto de IA). Mostra o relatório completo e, se alguma regra falhar, reprova a task do Job.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

from src.gold.validacao import levantar_se_houver_falhas, validar_gold

In [ ]:
# Notebook fino: as regras vivem em src/gold/validacao.py (testadas localmente).
resultados = validar_gold(
    dim_movies=spark.table("gold.dim_movies"),
    dim_genres=spark.table("gold.dim_genres"),
    dim_people=spark.table("gold.dim_people"),
    dim_companies=spark.table("gold.dim_companies"),
    dim_reviews=spark.table("gold.dim_reviews"),
    fact=spark.table("gold.fact_movies_performance"),
    bridge_genre=spark.table("gold.bridge_movie_genre"),
    bridge_person=spark.table("gold.bridge_movie_person"),
    bridge_company=spark.table("gold.bridge_movie_company"),
    contexto=spark.table("gold.gold_genai_movies_context"),
)

# Relatório completo (aprovadas e reprovadas), antes de decidir se a task falha.
display(
    spark.createDataFrame(
        [(r.regra, r.passou, r.detalhe) for r in resultados],
        "regra string, passou boolean, detalhe string",
    )
)

# Falha a task (e o Job) se qualquer regra foi reprovada.
levantar_se_houver_falhas(resultados)